# Reissner–Nordström black hole in Eddington–Finkelstein coordinates: gauge field and metric

Solves the coupled Einstein–Maxwell equations symbolically (SageMath manifolds) and identifies the extremal limit. The scalar-field perturbations and quasinormal modes are computed in `rn_charged_scalar_qnm.ipynb`.

In [1]:
%display latex

In [2]:
# useful function for simplifying the coefficients of a differential equation 
def simplify_coefficients(eqn,fcnlist,depvar,order):
    var('out,cf')
    out=0
    cf=0
    for j in range(len(fcnlist)):
        for k in range(order+1):
            cf = eqn.coefficient(diff(fcnlist[j],k)).full_simplify().expand()
            out = out + cf*diff(fcnlist[j],depvar,k)
    return out

In [3]:
BH = Manifold(4, 'BH', structure='Lorentzian')
print(BH)

4-dimensional Lorentzian manifold BH


In [4]:
H.<v,u,th,ph> = BH.chart(r"v u th:\vartheta ph:\varphi")
H

Chart (BH, (v, u, th, ph))

In [5]:
blackf = function('f')(u)
g = BH.metric()
g[0,0] = -blackf
g[0,1] = -1/u^2
g[2,2] = 1/u^2
g[3,3] = 1/u^2*sin(th)^2
g.display()

g = -f(u) dv⊗dv - 1/u^2 dv⊗du - 1/u^2 du⊗dv + u^(-2) dth⊗dth + sin(th)^2/u^2 dph⊗dph

In [6]:
Einstein = g.ricci() - 1/2*g*g.ricci_scalar()
Einstein.disp()

Ric(g)-r(g)*unnamed metric = (u^3*f(u)*d(f)/du - u^2*f(u)^2 + u^2*f(u)) dv⊗dv + (u*d(f)/du - f(u) + 1) dv⊗du + (u*d(f)/du - f(u) + 1) du⊗dv + 1/2*u^2*d^2(f)/du^2 dth⊗dth + 1/2*u^2*sin(th)^2*d^2(f)/du^2 dph⊗dph

In [7]:
Gauge = BH.one_form(name='Gauge', latex_name=r'\mathcal{A}')

In [8]:
potential=function('av')(u)

In [9]:
Gauge[:]=[potential,0,0,0]

In [10]:
F=Gauge.exterior_derivative()
F.disp()

dGauge = -d(av)/du dv∧du

In [11]:
Maxwelleqns= (F.hodge_dual(g)).exterior_derivative()
Maxwelleqns.disp()

d*dGauge = abs(sin(th))*d^2(av)/du^2 du∧dth∧dph

The solution to the Maxwell equations is easy. 
We chose a gauge such that the potential vanishes at $u=1$ and has charge $Q$. Later we also fix the horizon to sit at $u=1$.

In [12]:
var('Q')
avsol(u) = Q*(u-1) 

In [13]:
(Maxwelleqns[1,2,3].expr()).substitute_function(av,avsol)

0

We need the energy momentum tensor of the Maxwell field as source for Einsteins equations:

In [14]:
Lagdensity = -1/2*F.wedge(F.hodge_dual(g))

In [15]:
Lagdensity.disp()

1/2*abs(sin(th))*(d(av)/du)^2 dv∧du∧dth∧dph

In [16]:
# Since in form language we directly compute the Lagrangian density 
# we get the Lagrangian through dividing by the volume element
sqrtg=g.volume_form()[0,1,2,3] #volume element 
Lag =  Lagdensity[0,1,2,3]/sqrtg
Lag

1/2*u^4*(d(av)/du)^2

The energy momentum tensor for the Maxwell field is 
$$ T_{\mu\nu}= F_{\mu\alpha} F_\nu\,^\alpha + g_{\mu\nu} \mathcal{L}$$

In [17]:
Tem = F.contract(1,F.up(g,1)) + g*Lag
#Amb = D
Tem.disp()

1/2*u^4*f(u)*(d(av)/du)^2 dv⊗dv + 1/2*u^2*(d(av)/du)^2 dv⊗du + 1/2*u^2*(d(av)/du)^2 du⊗dv + 1/2*u^2*(d(av)/du)^2 dth⊗dth + 1/2*u^2*sin(th)^2*(d(av)/du)^2 dph⊗dph

In [18]:
var('κ')
eomEinMax=(Einstein-1/κ*Tem)[0,1].expr().substitute_function(av,avsol).expand()
eomEinMax

-1/2*Q^2*u^2/κ + u*diff(f(u), u) - f(u) + 1

We solve this differential equation and adjust the integration constant such that f(1)=0 .

In [19]:
fsol(u)=desolve( eomEinMax ,[f(u),u],ics=[1,0]).expand()

In [20]:
fsol

u |--> 1/2*Q^2*u^2/κ - 1/2*Q^2*u/κ - u + 1

In [21]:
F[0,1].expr().substitute_function(av,avsol)

-Q

This is the electric field, it looks as if it is independent of the coordinate u! But note that F is a 2-form.
We have $F= -Q dt\wedge du $ and with $u=1/r$ and $du = -\frac{dr}{r^2}$ that means $F=\frac{Q}{r^2} dt\wedge dr$ and thus $E_r = \frac{Q}{r^2}$. Indeed the black hole looks like a point charge of strength $Q$ for $u>1$.

In [22]:
(Einstein-1/κ * Tem).disp()

-1/2*(u^4*f(u)*(d(av)/du)^2 - 2*(u^3*f(u)*d(f)/du - u^2*f(u)^2 + u^2*f(u))*κ)/κ dv⊗dv - 1/2*(u^2*(d(av)/du)^2 - 2*(u*d(f)/du - f(u) + 1)*κ)/κ dv⊗du - 1/2*(u^2*(d(av)/du)^2 - 2*(u*d(f)/du - f(u) + 1)*κ)/κ du⊗dv - 1/2*(u^2*(d(av)/du)^2 - u^2*κ*d^2(f)/du^2)/κ dth⊗dth - 1/2*(u^2*sin(th)^2*(d(av)/du)^2 - u^2*κ*sin(th)^2*d^2(f)/du^2)/κ dph⊗dph

We check if indeed all equations are solved:

In [23]:
[ 
    [
    (Einstein-1/κ * Tem)[i,j].expr().substitute_function(av,avsol).substitute_function(f,fsol).expand()
    for j in range(0,4)]
for i in range(0,4) ]

[[0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]]

In [24]:
fsol(u).subs(Q=sqrt(2*κ)).factor()

(u - 1)^2

Note: For $Q=\sqrt{2\kappa}$ we get a double zero on the horizon! This is a so-called degenerate horizon and the black hole is called "extremal". It is rather special, for larger $Q$ values the singularity lies outside the horizon. This is a situation we need to avoid ("naked singularity"). So we have to restrict the allowed values to $0 \le Q \le \sqrt{2\kappa}$.